# Start the nim for genMOL

**starts on Port 8083**


In [1]:
import boto3
import json

# Replace with your secret name and region
secret_name = 'NVIDIA_NGC_CREDS_VINCENT'
region_name = 'us-east-1'

# Create a Secrets Manager client
client = boto3.client('secretsmanager', region_name=region_name)

# Retrieve the secret value
response = client.get_secret_value(SecretId=secret_name)
secret = json.loads(response['SecretString'])

# Store the API key in a Python variable
ngc_cli_api_key = secret['NGC_CLI_API_KEY']

# (Optional) Print a success message (avoid printing the actual key)
print("API key retrieved successfully.")

API key retrieved successfully.


In [2]:
%%bash -s "$ngc_cli_api_key"
# Set the API key as an environment variable
export NGC_CLI_API_KEY=$1

# Authenticate to your ECR registry
aws ecr get-login-password --region us-east-1 | docker login --username AWS --password-stdin 841162696343.dkr.ecr.us-east-1.amazonaws.com

# NVCR login
docker login nvcr.io --username '$oauthtoken' --password $NGC_CLI_API_KEY

# Run the container from your ECR repository
docker run -d --rm --name genmol-prod --runtime=nvidia \
    -e CUDA_VISIBLE_DEVICES=0 \
    -e NGC_API_KEY=$NGC_CLI_API_KEY \
    -p 8083:8000 \
    841162696343.dkr.ecr.us-east-1.amazonaws.com/genmol:latest

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded


WARNING! Using --password via the CLI is insecure. Use --password-stdin.
WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
a592c3bd9ea2c473b58ab216d323e52d6d570a3271c6ae7fa321cb7a4cd025ba


## check if  its running

In [3]:
!docker ps -a
# In case you want to stop the container -> !docker stop genmol

CONTAINER ID   IMAGE                                                        COMMAND                  CREATED         STATUS         PORTS                                       NAMES
a592c3bd9ea2   841162696343.dkr.ecr.us-east-1.amazonaws.com/genmol:latest   "/bin/bash -c $SERVE…"   3 minutes ago   Up 2 minutes   0.0.0.0:8083->8000/tcp, :::8083->8000/tcp   genmol-prod


## stop the container after use

In [4]:
!docker stop genmol-prod

genmol-prod
